Author: Danny Lillard

Date: 2026-04-03

Desc: Testing return of algorithm we will look at the following:

- Buy and hold
- Naive algo
- Clustering algo

In [1]:
# buy and hold
import pandas as pd

raw_data_df = pd.read_csv("quarterly_clusters_model/aa_daily_data.csv")
raw_data_df['timestamp'] = pd.to_datetime(raw_data_df['timestamp'])

start_date = pd.to_datetime("2022-04-01")
end_date = pd.to_datetime("2022-06-30")

start_and_stop_df = raw_data_df[raw_data_df['timestamp'].isin([start_date, end_date])]

# Get the unique tickers in the dataset
tickers = start_and_stop_df['ticker'].unique()

# Calculate the return for each ticker
returns = []
for ticker in tickers:
    ticker_data = start_and_stop_df[start_and_stop_df['ticker'] == ticker]
    ticker_data = ticker_data.sort_values('timestamp')
    
    # Get the opening price on the start date and the closing price on the end date
    start_price = ticker_data[ticker_data['timestamp'] == start_date]['open'].values[0]
    end_price = ticker_data[ticker_data['timestamp'] == end_date]['close'].values[0]
    
    # Calculate the return
    return_pct = (end_price - start_price) / start_price
    returns.append((ticker,return_pct))

# save returns to csv
returns_df = pd.DataFrame(returns, columns=['ticker', 'return'])
returns_df.to_csv("buy_and_hold_returns.csv", index=False)

# calculate overall return
overall_return = sum([r[1] for r in returns]) / len(returns)
print(f"Overall Buy and Hold Return: {overall_return:.2%}")

Overall Buy and Hold Return: -11.03%


In [ ]:
# # zipping returns for clustering algo

# clustering_signals = pd.read_csv('quarterly_clusters_model/signals_and_returns.csv', header=None)

# clustering_signals.columns = ['timestamp', 'ticker', 'signal']

# clustering_signals['timestamp'] = pd.to_datetime(clustering_signals['timestamp'])

# raw_data_df = pd.read_csv("quarterly_clusters_model/aa_daily_data.csv")
# raw_data_df['timestamp'] = pd.to_datetime(raw_data_df['timestamp'])

# clustering_signals_and_returns = pd.merge(clustering_signals, raw_data_df, on=['timestamp', 'ticker'], how='left')

# def compute_trade_return(signal: str, open_t: float, close_t: float) -> float:
#     if signal == "BUY":
#         return (close_t - open_t) / open_t
#     elif signal == "SELL":
#         return (open_t - close_t) / open_t
#     return 0.0

# clustering_signals_and_returns['return'] = clustering_signals_and_returns.apply(lambda row: compute_trade_return(row['signal'], row['open'], row['close']), axis=1)


In [2]:
clustering_signals_and_returns = pd.read_csv('quarterly_clusters_model/q2_2022_clustered_signals_and_returns.csv', header=None)

clustering_signals_and_returns.columns = ['timestamp', 'ticker', 'signal', 'return']

clustering_signals_and_returns['timestamp'] = pd.to_datetime(clustering_signals_and_returns['timestamp'])

clustering_signals_and_returns['return'].groupby(clustering_signals_and_returns['ticker']).mean().sum()

np.float64(0.004381849256026466)

In [3]:
# seeing returns for naive algo
naive_returns_df = pd.read_csv("naive_2022_apr_jun_signals_and_returns.csv")

naive_returns_df.columns = ['timestamp', 'ticker', 'signal', 'return']

overall_naive_return = naive_returns_df['return'].groupby(naive_returns_df['ticker']).mean().sum()
overall_naive_return

np.float64(-0.04919176266631502)

In [4]:
# success rate
clustering_signals_and_returns['success'] = clustering_signals_and_returns.apply(lambda row: 1 if row['return'] > 0 else 0, axis=1)

success_rate = clustering_signals_and_returns['success'].mean()
print(f"Clustering Algorithm Success Rate: {success_rate:.2%}")

Clustering Algorithm Success Rate: 44.59%


In [7]:
# what tickers lost money?

clustering_signals_and_returns.groupby('ticker')['return'].sum().sort_values().head(30)

# for q2_2022 the worst stocks greatly impacted returns.

ticker
ARAY   -0.676321
TMDX   -0.560680
WRAP   -0.376753
CMP    -0.355644
OKTA   -0.346441
BDN    -0.332421
EARN   -0.319793
STRA   -0.292802
OMF    -0.286849
TPC    -0.260751
BDC    -0.237370
FNKO   -0.232092
VAC    -0.221007
EXPE   -0.201407
ANET   -0.197975
PRTA   -0.191131
TFSL   -0.188289
CPS    -0.188244
WPC    -0.182349
OPK    -0.174625
VFC    -0.170331
GE     -0.169429
WEYS   -0.151986
VRTX   -0.148903
NWL    -0.142079
TEAM   -0.133685
AMD    -0.133669
NNBR   -0.132747
DD     -0.132536
HCSG   -0.131211
Name: return, dtype: float64

In [6]:
# looking at 30 worst stocks with a twice as big training period

worst_df = pd.read_csv("quarterly_clusters_model/q2_2022_worst_stocks_clustered_signals_and_returns.csv")

worst_df.columns = ['timestamp', 'ticker', 'signal', 'return']

worst_df['timestamp'] = pd.to_datetime(worst_df['timestamp'])

worst_df.groupby('ticker')['return'].sum().sort_values()

ticker
ARAY   -0.909794
CMP    -0.469846
WRAP   -0.376753
OKTA   -0.346441
EARN   -0.273353
TPC    -0.260751
ANET   -0.197975
TFSL   -0.188289
CPS    -0.188244
EXPE   -0.172878
VFC    -0.170331
BDN    -0.159338
OPK    -0.150935
VRTX   -0.148903
VAC    -0.146323
NNBR   -0.119760
WEYS   -0.102505
PRTA   -0.093312
HCSG   -0.077254
TEAM   -0.076284
WPC    -0.063212
DD     -0.008543
STRA   -0.007983
GE      0.013366
OMF     0.052470
TMDX    0.102216
BDC     0.130483
AMD     0.175581
NWL     0.245869
FNKO    0.259426
Name: return, dtype: float64

In [8]:
worst_df.groupby('ticker')['return'].sum().mean()

np.float64(-0.12431988489712677)

In [10]:
clustering_signals_and_returns.groupby('ticker')['return'].head(30).sum().mean()

np.float64(-1.5138150258641476)